# 🧪 atomipy Visual Builder - Google Colab GPU Launch Guide
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mholmboe/atomipy-web-module/blob/main/ColabLaunchGuide.ipynb)

Welcome to the official launch manual for running the **atomipy Visual Builder** on **Google Colab** with full **GPU hardware acceleration**!

This notebook launches the visual interface in Colab's cloud environment so you can build complex mineral-water systems and run **OpenMM molecular dynamics simulations** on a high-performance **NVIDIA T4, L4, or A100 GPU**.

Unlike the public site at [www.atomipy.io](https://www.atomipy.io) (which runs on CPU only), GPU simulations are **enabled** here.

---

## ⚡ STEP 0: Enable GPU Acceleration

1. In the top-right menu of Colab, click **Runtime** -> **Change runtime type**.
2. Select **T4 GPU** (or a higher tier if available) under *Hardware accelerator*.
3. Click **Save**.

## 📦 STEP 1: Clone and Build the Application

This wipes any previous install, clones the repo, installs the Python dependencies (FastAPI + OpenMM + atomipy deps), and builds the React/Vite production frontend bundle.

In [ ]:
# 1. Reset directory and wipe previous folders
%cd /content
!rm -rf atomipy-web-module

# 2. Clone fresh
!git clone https://github.com/mholmboe/atomipy-web-module.git
%cd atomipy-web-module

# 3. Install Python dependencies (FastAPI backend + OpenMM + atomipy deps)
!pip install -q -r requirements.txt

# 4. Install Node and build the React frontend (served by FastAPI)
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!npm install --legacy-peer-deps
!npm run build

# 5. Cloudflare Quick Tunnel binary for the public URL (free, no account/login)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

## 🧪 STEP 1b (OPTIONAL): Enable the Organic Molecule (GAFF/OpenFF) node

The **Organic Molecule** node parametrizes small molecules with GAFF / OpenFF
via a separate **OpenFF worker**. Run the cell below **only if you need that
node** — it installs the OpenFF + ACPYPE stack with **micromamba** (no kernel
restart) and starts the worker on port `8001`. It adds a few minutes the first
time. **Skip it** if you only build systems, assign MINFF/CLAYFF, and run MD/EM.

> Run this **before** Step 2. All other nodes work without it.

In [ ]:
# (Optional) OpenFF/ACPYPE worker for the Organic Molecule node.
import os, subprocess, time, json, urllib.request

REPO = "/content/atomipy-web-module"
os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba"
MAMBA = "/content/bin/micromamba"

# 1. Standalone micromamba binary (no conda install, no kernel restart)
if not os.path.exists(MAMBA):
    subprocess.run(
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest "
        "| tar -xj -C /content bin/micromamba",
        shell=True, check=True,
    )

# 2. Create the atomipy-openff environment (openff-toolkit, acpype, rdkit, openbabel, ...)
env_python = "/content/micromamba/envs/atomipy-openff/bin/python"
if not os.path.exists(env_python):
    print("Installing OpenFF/ACPYPE stack - this takes a few minutes...")
    subprocess.run(
        [MAMBA, "create", "-y", "-f", f"{REPO}/envs/atomipy-openff.yml"],
        check=True,
    )

# 3. Launch the OpenFF worker on 127.0.0.1:8001 in the background
wenv = os.environ.copy()
wenv["PYTHONPATH"] = f"{REPO}/workers/openff_worker"
wenv["INTERCHANGE_EXPERIMENTAL"] = "1"
subprocess.Popen(
    [MAMBA, "run", "-n", "atomipy-openff",
     "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8001"],
    cwd=f"{REPO}/workers/openff_worker", env=wenv,
    stdout=open("/content/openff_worker.log", "w"), stderr=subprocess.STDOUT,
)

# 4. Wait for the worker to be ready (the main server uses OPENFF_WORKER_URL,
#    which defaults to http://127.0.0.1:8001 - no extra config needed).
for _ in range(90):
    try:
        s = urllib.request.urlopen("http://127.0.0.1:8001/status", timeout=2).read()
        print("\u2705 OpenFF worker ready (acpype_available =",
              json.loads(s).get("acpype_available"), ")")
        break
    except Exception:
        time.sleep(3)
else:
    print("\u26a0\ufe0f Worker not up yet - check /content/openff_worker.log")

## ⚛️ STEP 1c (OPTIONAL): Enable the GROMACS engine on the GPU

The **Simulate** node can run either OpenMM or **GROMACS**. OpenMM works out of
the box; to use the **GROMACS** engine, run the cell below **before Step 2**. It
installs a **CUDA-enabled GROMACS** with **micromamba** (no kernel restart) and
puts `gmx` on PATH, so grompp + mdrun run on the Colab GPU. It adds a few minutes
the first time. **Skip it** if you only use the OpenMM engine.

> After it finishes: in the **GROMACS** Simulate node, **clear the "GROMACS path"
> field** so it uses `gmx` on PATH. Then run as usual (chain EM → NVT → NPT).

In [ ]:
# (Optional) Enable the GROMACS engine on the Colab GPU.
# Installs a CUDA-enabled GROMACS with micromamba (no kernel restart) and puts
# `gmx` on PATH so the app's GROMACS Simulate node runs grompp + mdrun on the GPU.
import os, subprocess

os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba"
MAMBA = "/content/bin/micromamba"

# 1. Standalone micromamba binary (shared with Step 1b; no conda, no kernel restart)
if not os.path.exists(MAMBA):
    subprocess.run(
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest "
        "| tar -xj -C /content bin/micromamba",
        shell=True, check=True,
    )

# 2. Create the atomipy-gromacs env with a CUDA GROMACS (falls back to a plain
#    build if the cuda selector is unavailable on this Colab image).
GMX_BIN = "/content/micromamba/envs/atomipy-gromacs/bin"
if not os.path.exists(f"{GMX_BIN}/gmx"):
    print("Installing CUDA GROMACS - this takes a few minutes...")
    rc = subprocess.run(
        [MAMBA, "create", "-y", "-n", "atomipy-gromacs", "-c", "conda-forge",
         "gromacs=*=nompi*cuda*"],
    ).returncode
    if rc != 0:
        subprocess.run(
            [MAMBA, "create", "-y", "-n", "atomipy-gromacs", "-c", "conda-forge", "gromacs"],
            check=True,
        )

# 3. Put gmx (and its libs) on PATH for the server launched in Step 2, which
#    inherits os.environ via os.environ.copy().
os.environ["PATH"] = GMX_BIN + ":" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = (
    "/content/micromamba/envs/atomipy-gromacs/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
)

# 4. Verify gmx + GPU.
v = subprocess.run([f"{GMX_BIN}/gmx", "--version"], capture_output=True, text=True)
print("\n".join(l for l in v.stdout.splitlines() if "version" in l.lower())[:200] or v.stdout[:200])
gpu = subprocess.run(["bash", "-lc", "nvidia-smi -L"], capture_output=True, text=True).stdout
print(gpu or "\u26a0\ufe0f No GPU detected - set Runtime type to GPU and rerun.")
print("\u2705 GROMACS ready. In the GROMACS Simulate node, CLEAR the 'GROMACS path' "
      "field so it uses 'gmx' on PATH.")

## 🚀 STEP 2: Launch the Visual Builder

This cell:
1. Starts a free **Cloudflare Quick Tunnel** (no account, no login, **no password**).
2. Boots the **FastAPI** server on port `5002`, which serves both the API *and* the built frontend, with simulations **enabled** (GPU).

### Instructions
- 🔗 **Click the `https://….trycloudflare.com` link** printed below — that's it, no password needed.
- The first load can take a few seconds while the tunnel and server warm up.

> The **Organic Molecule (GAFF/OpenFF)** node works here only if you ran the optional **Step 1b** above; otherwise that single node is unavailable (everything else still works).

In [ ]:
import os
import subprocess
import re

REPO = "/content/atomipy-web-module"
PORT = "5002"

# 1. Start a free Cloudflare Quick Tunnel in the background (no account/login).
#    It exposes a public *.trycloudflare.com URL and proxies it to the local
#    server, retrying until the server (started below) comes up.
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
for line in cf_proc.stdout:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

print("\n==================================================")
print(f"👉 OPEN THIS LINK: {public_url}")
print("   (no password needed)")
print("==================================================\n")

# 2. Launch the FastAPI server (serves frontend + API).
#    SIMULATION_MODE is left unset, which defaults to 'full', so Energy
#    Minimization and NVT/NPT MD all run on the Colab GPU.
#    OPENFF_WORKER_URL defaults to http://127.0.0.1:8001 (the Step 1b worker).
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO}:{REPO}/backend/core"
env["FRONTEND_DIST"] = f"{REPO}/dist"
subprocess.run(
    [
        "uvicorn", "main:app",
        "--app-dir", f"{REPO}/backend/core",
        "--host", "0.0.0.0",
        "--port", PORT,
    ],
    env=env,
)